# 🎬 مولّد فيديوهات الإعلانات — Wan 2.1 (مجاني + مفتوح المصدر)

يشغّل على كولابك أفضل نموذج فيديو مفتوح المصدر ممكن يعمل على كارت T4 المجاني: **Wan 2.1 من Alibaba** (نسخة 1.3B).

## وش بتقدر تسوي فيه؟
- **نص → فيديو**: تكتب قصة إنكليزي ويطلع لك مقطع إعلاني متحرك
- **صورة → فيديو**: تحوّل بوستر/شعار/صورة منتج لمقطع بحركة سينمائية
- مقاطع **3–5 ثواني** بدقة 480p (افقي أو عمودي للتيكنز والريلز)

## قبل ما تبدأ
1. من قائمة Colab: **Runtime → Change runtime type → T4 GPU**
2. اضغط `Shift+Enter` على الخانات **بالترتيب**
3. كل مقطع ياخد **دقائق** على T4 — هذا طبيعي، الفيديو يعلّق.UI

> 💡 **حقوقك:** الرخص Apache 2.0 → مقاطعك **ملكك 100% وتقدر تستخدمها بإعلانات ممولة رسمياً** — بدون علامة مائية وبلا رسوم.

> 📦 **مساحته:** التحميل الكامل **~27GB** (المشفر النصي 21GB + المولد 5GB) — ينزل مرة واحدة فقط وياخذ 10-20 دقيقة أول صف.
> ⚠️ **ذاكرة كولاب المجاني محدودة (~13GB):** إذا طلعت أخطاء `Out of Memory` عند التحميل → فيه نسخة أخف (GGUF ~10GB) مضمونة على T4، قلّي بس وأحولك لها.
> ⚠️ كولاب المجاني قد ينقطع بعد ساعات — ولّد مقاطعك وخزّنها على جهازك مباشرة.

In [ ]:
# ===== الخطوة 1: التثبيت (مرة واحدة لكل جلسة) =====
print("GPU المتاح:")
!nvidia-smi -L

# مكتبات تشغيل نماذج الفيديو + حفظ MP4
!pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece imageio imageio-ffmpeg opencv-python-headless
print("\nالتثبيت تم ✅")

In [ ]:
# ===== الخطوة 2: تحميل النموذج (الأفضل جودةً على T4) =====
import torch
from diffusers import WanPipeline

# Wan 2.1 من Alibaba — مفتوح المصدر (Apache 2.0)، بلا علامة مائية، وجاهز للاستخدام التجاري
MODEL = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print("⏳ تحميل النموذج... أول مرة ياخذ 10-20 دقيقة (نحو 27GB — أغلبها المشفر النصي)")
pipe = WanPipeline.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()   # يوفّر الذاكرة على T4
pipe.enable_vae_tiling()
pipe.enable_vae_slicing()
print("✅ النموذج جاهز!")

In [ ]:
# ===== الخطوة 3: توليد فيديو إعلاني (نص → فيديو) =====
import numpy as np, imageio.v2 as imageio
from IPython.display import display, Video
from google.colab import files

# ✍️ اكتب قصّتك بالإنجليزي (النتائج أفضل بكثير) — مثال جاهز للموقع:
prompt = "A sleek smartphone on a desk, an elegant Arabic AI chat app answering questions instantly, golden and deep blue lighting, cinematic slow zoom, premium tech commercial"

negative = "blurry, low quality, low resolution, watermark, text, subtitle, distortion, static"

# 📐 الاتجاه: سطر واحد فقط
W, H = 832, 480        # افقي (16:9) ← للـ YouTube والفيس بوك
# W, H = 480, 832      # عمودي ← للتيكنز والريلز (أحذف # من أول السطر)

print(f"⏳ جاري توليد الفيديو ({W}x{H})... يستغرق دقائق على T4")
frames = pipe(
    prompt=prompt,
    negative_prompt=negative,
    width=W, height=H,
    num_frames=81,          # ~5 ثواني بمعدل 16 صورة/ثانية
    num_inference_steps=30,
    guidance_scale=5.0,
).frames[0]

# حفظ MP4
SAVE = "/content/ad_video.mp4"
writer = imageio.get_writer(SAVE, fps=16, codec="libx264", quality=8)
for f in frames:
    writer.append_data(np.array(f))
writer.close()

print("✅ تم التوليد:", SAVE)
display(Video(SAVE))
files.download(SAVE)      # ينزّل المقطع على جهازك

## ✍️ نصوص جاهزة لإعلاناتك (انسخ أحدها فوق في `prompt`)

| المقطع | النص (الصقة بالإنجليزي) |
|---|---|
| **تعريفي** | *A premium smartphone on a modern desk, an elegant Arabic AI chat app answering questions, golden and deep blue lighting, cinematic slow push-in, ultra sharp, premium commercial* |
| **مقارنة** | *Split screen: stressed person drowning in tasks on the left, relaxed person using a smart Arabic AI assistant on the right, clean bright advertising style, smooth transition* |
| **إبهار** | *Abstract streams of glowing Arabic calligraphy data forming interface windows in the air, futuristic fintech aesthetic, slow motion golden particles, elegant* |
| **حل سريع** | *Person speaks to a glowing AI orb that projects beautiful artwork and photos into the air, dark studio with neon blue and gold accents, cinematic camera orbit* |
| **شهادات عيد** | *Best prompts, best outcomes, feedback, great AI answers, high contrast product commercial, single product focus, macro detail, luxurious lighting* |

**سرّ الجودة:** زوّد نصوصك بالحركة والكاميرا: `slow zoom`, `camera orbit`, `gentle pan`, `cinematic lighting` — وقلّل عدد الكلمات الزائدة.

بعد التوليد: ضف **شعار الموقع + النص الإعلاني** بمونتاج CapCut أو Canva (مجانيان) وانشر. 🚀

In [ ]:
# ===== (اختياري) صورة ← فيديو: حرّك بوستر/صورة صممتها =====
# مفيد جداً: صمّم بوستر بالقيم ثم حرّكه لين مقطع إعلاني من غير ما تكتب مشهد
import torch
from PIL import Image
import numpy as np, imageio.v2 as imageio
from IPython.display import display, Video
from google.colab import files as _files
from diffusers import WanImageToVideoPipeline

print("⏳ تحميل نموذج الصورة→فيديو... دقائق")
pipe2 = WanImageToVideoPipeline.from_pretrained("Wan-AI/Wan2.1-I2V-1.3B-Diffusers", torch_dtype=torch.bfloat16)
pipe2.enable_model_cpu_offload()
pipe2.enable_vae_tiling()

# 📤 ارفع صورتك (بوستر / شعار / صورة منتج)
up = _files.upload()
img = Image.open(list(up.keys())[0]).convert("RGB")

# قصّ لمقاس 832x480 مع المحافظة على شكل الصورة
W, H = 832, 480
scale = max(W / img.width, H / img.height)
img = img.resize((int(img.width * scale) + 1, int(img.height * scale) + 1), Image.LANCZOS)
img = img.crop(((img.width - W) // 2, (img.height - H) // 2,
                (img.width - W) // 2 + W, (img.height - H) // 2 + H))

motion = "cinematic slow zoom in, gentle camera move, smooth, subtle"
print("⏳ توليد الحركة على صورتك... دقائق")
frames = pipe2(
    image=img,
    prompt=motion,
    negative_prompt="blurry, low quality, watermark, distortion, static",
    width=W, height=H,
    num_frames=81,
    num_inference_steps=30,
    guidance_scale=5.0,
).frames[0]

SAVE = "/content/ad_from_image.mp4"
writer = imageio.get_writer(SAVE, fps=16, codec="libx264", quality=8)
for f in frames:
    writer.append_data(np.array(f))
writer.close()
print("✅ تم التوليد:", SAVE)
display(Video(SAVE))
_files.download(SAVE)

## 📌 خلاصة سريعة
- **كل مقطع:** 3–5 ثواني، دقائق توليد على T4، بدقة 480p، بدون علامة مائية
- **استخدامك التجاري:** مسموح (Apache 2.0) — مقاطعك ملكك
- **الأسلوب الاحترافي:** ولّد 5–10 مقاطع → ضف الشعار والنص في CapCut/Canva → ثبّت مرة كل يوم
- **كولاب ينقطع:** خزّن كل MP4 فور توليده على جهازك (الخانة تنزّله تلقائياً)

> 🧠 قدرة محترفة أكبر (وضوح أعلى + مقاطع أطول) = أدوات سحابية مثل Kling/Veo المجانية اليومية — لكن هذا الخيار **لا نهائي ومجاني 100% وملكك**.